# Quant Risk Core: Comprehensive Analysis Suite
This notebook provides a deep-dive into every module of the `quant_risk_core` repository, covering derivatives pricing, volatility modeling, portfolio risk, and credit risk.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import scipy.stats as stats

## 1. Derivatives Pricing: Black-Scholes & Greeks
The Black-Scholes model provides the theoretical price of European options and their sensitivities (Greeks).

In [2]:
from market_risk.pricing import BlackScholesEngine

S = np.linspace(80, 120, 100)
K, T, r, sigma = 100, 1.0, 0.05, 0.2

calls = BlackScholesEngine.calculate_prices(S, K, T, r, sigma, 'call')
puts = BlackScholesEngine.calculate_prices(S, K, T, r, sigma, 'put')

fig_bs = go.Figure()
fig_bs.add_trace(go.Scatter(x=S, y=calls, name='Call Price'))
fig_bs.add_trace(go.Scatter(x=S, y=puts, name='Put Price'))
fig_bs.update_layout(title='Black-Scholes Option Prices vs Spot', xaxis_title='Spot Price', yaxis_title='Option Value')
fig_bs.show()

# Sensitivity Analysis: Greeks
greeks = [BlackScholesEngine.calculate_greeks(s, K, T, r, sigma) for s in S]
deltas = [g['Delta'] for g in greeks]
gammas = [g['Gamma'] for g in greeks]

fig_greeks = make_subplots(rows=1, cols=2, subplot_titles=('Delta', 'Gamma'))
fig_greeks.add_trace(go.Scatter(x=S, y=deltas, name='Delta'), row=1, col=1)
fig_greeks.add_trace(go.Scatter(x=S, y=gammas, name='Gamma'), row=1, col=2)
fig_greeks.update_layout(title='Option Greeks Sensitivity')
fig_greeks.show()

## 2. Advanced Volatility: EGARCH & Regime Switching
We analyze asymmetric volatility effects and market regime detection.

In [3]:
from market_risk.volatility import GARCHEngine, RegimeSwitchingEngine

np.random.seed(42)
returns = np.random.normal(0, 0.01, 1000)
returns[500:600] *= 5 # High vol regime
returns_series = pd.Series(returns)

# EGARCH analysis
egarch = GARCHEngine()
egarch.fit(returns_series, model_type='EGARCH')
print(f'EGARCH Gamma (Leverage Effect): {egarch.gamma:.4f}')

# Regime Switching analysis
regime_eng = RegimeSwitchingEngine(k_regimes=2)
regime_eng.fit(returns_series)
probs = regime_eng.get_regime_probabilities()

fig_regime = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, subplot_titles=('Returns', 'High Vol Regime Probability'))
fig_regime.add_trace(go.Scatter(y=returns, name='Returns'), row=1, col=1)
fig_regime.add_trace(go.Scatter(y=probs.iloc[:, 1], name='Regime 1 Prob', fill='tozeroy'), row=2, col=1)
fig_regime.update_layout(height=600, title='Markov Regime Switching Analysis')
fig_regime.show()

EGARCH Gamma (Leverage Effect): 0.0000


## 3. Portfolio Risk: Copulas & Decomposition
How risk is distributed across a portfolio of correlated assets.

In [4]:
from portfolio_risk.decomposition import RiskDecomposer, CopulaEngine

weights = np.array([0.4, 0.3, 0.3])
cov = np.array([
    [0.0004, 0.0001, 0.0002],
    [0.0001, 0.0009, 0.0003],
    [0.0002, 0.0003, 0.0006]
])

decomposer = RiskDecomposer(weights, cov)
mvar = decomposer.calculate_marginal_var(0.99)
cvar = decomposer.calculate_component_var(0.99)

fig_risk = go.Figure(data=[go.Pie(labels=['Asset A', 'Asset B', 'Asset C'], values=cvar, hole=.3)])
fig_risk.update_layout(title='Component VaR Distribution (Risk Contribution)')
fig_risk.show()

# Copula dependency visualization
corr = np.array([[1.0, 0.8], [0.8, 1.0]])
copula = CopulaEngine(corr)
samples = copula.generate_gaussian_copula_samples(2000)
fig_copula = px.scatter(x=samples[:, 0], y=samples[:, 1], opacity=0.5, title='Gaussian Copula Dependency Structure')
fig_copula.show()

## 4. Liquidity-Adjusted Risk & Stress Testing
Modeling market friction and extreme scenarios.

In [5]:
from market_risk.liquidity import LiquidityRiskEngine
from market_risk.stress_testing import FactorStresser

liquidity = LiquidityRiskEngine(position_size=5000, mid_price=100)
spreads = np.linspace(0.001, 0.02, 20)
l_vars = [liquidity.calculate_l_var(10000, s) for s in spreads]

fig_liq = go.Figure()
fig_liq.add_trace(go.Scatter(x=spreads, y=l_vars, name='L-VaR'))
fig_liq.update_layout(title='L-VaR Sensitivity to Bid-Ask Spread', xaxis_title='Spread', yaxis_title='Adjusted VaR')
fig_liq.show()

# Correlation stress
corr_orig = np.array([[1.0, 0.3], [0.3, 1.0]])
stressed_corr = FactorStresser.tilt_correlation(corr_orig, 2.5)
print('Original Correlation:\n', corr_orig)
print('Stressed Correlation (2.5x tilt):\n', stressed_corr)

Original Correlation:
 [[1.  0.3]
 [0.3 1. ]]
Stressed Correlation (2.5x tilt):
 [[1.   0.75]
 [0.75 1.  ]]


## 5. Credit Risk: WWR, CVA & Migration
Analyzing counterparty credit transitions and correlation risk.

In [6]:
from credit_risk.counterparty import RatingMigrationEngine, CounterpartyRiskEngine

# Rating Migration paths
tm = np.array([[0.9, 0.08, 0.02], [0.1, 0.8, 0.1], [0.05, 0.15, 0.8]])
mig_eng = RatingMigrationEngine(tm)
n_paths = 5
fig_mig = go.Figure()
for i in range(n_paths):
    path = mig_eng.simulate_migration(0, 20)
    fig_mig.add_trace(go.Scatter(y=path, mode='lines+markers', name=f'Counterparty {i+1}'))
fig_mig.update_layout(title='Simulated Credit Rating Migration Paths', yaxis=dict(tickvals=[0, 1, 2], ticktext=['AAA', 'BBB', 'CCC']))
fig_mig.show()

# WWR Impact
t_grid = np.linspace(0, 1, 10)
cpty = CounterpartyRiskEngine(t_grid)
cpty.set_portfolio_paths(np.random.normal(0, 10, (100, 10)))
pd_curv = np.linspace(0, 0.05, 10)
alphas = np.linspace(1.0, 1.5, 10)
wwr_cvas = [cpty.calculate_cva_wwr(0.4, pd_curv, a) for a in alphas]

fig_wwr = go.Figure()
fig_wwr.add_trace(go.Scatter(x=alphas, y=wwr_cvas))
fig_wwr.update_layout(title='CVA Sensitivity to Wrong-Way Risk Alpha', xaxis_title='Alpha Multiplier', yaxis_title='Adjusted CVA')
fig_wwr.show()

## 6. Real-time Market Data Integration
Fetching live data using the Yahoo Finance connector.

In [7]:
from data.data_connectors import YahooFinanceConnector

try:
    prices = YahooFinanceConnector.fetch_historical_prices(['SPY', 'TLT'], '2023-01-01', '2024-01-01')
    fig_data = px.line(prices, title='Market Data Fetch: SPY vs TLT')
    fig_data.show()
except Exception as e:
    print(f'Data fetch skipped (possibly network/API limits): {e}')